In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# =========================================================
# ASSUMPTIONS
# =========================================================
# Existing DataFrames:
#
df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")
df_po = pd.read_csv("data/raw/df_purchase_orders.csv")
df_shipments = pd.read_csv("data/raw/df_shipments.csv")  
#
# Recommended shipment columns:
# shipment_id
# po_id
# supplier_id
# shipment_status
# actual_arrival
#
# Recommended PO columns:
# po_id
# received_quantity
# order_value
#
# Because enterprise traceability matters.
# Otherwise auditors begin materializing from walls.
# =========================================================


# =========================================================
# DEFECT TYPES BY SUPPLIER CATEGORY
# =========================================================

defect_map = {

    "electronics": [
        "PCB Damage",
        "Component Misalignment",
        "Solder Failure",
        "ESD Damage",
        "Voltage Instability"
    ],

    "metals": [
        "Surface Corrosion",
        "Dimensional Variance",
        "Cracks",
        "Material Impurity"
    ],

    "packaging": [
        "Seal Failure",
        "Compression Damage",
        "Print Misalignment"
    ],

    "chemicals": [
        "Contamination",
        "Incorrect Composition",
        "Moisture Exposure"
    ],

    "plastics": [
        "Warping",
        "Color Mismatch",
        "Structural Weakness"
    ],

    "logistics": [
        "Handling Damage",
        "Labeling Error"
    ],

    "textiles": [
        "Fabric Tear",
        "Color Bleeding",
        "Stitch Defect"
    ]
}


# =========================================================
# SUPPLIER QUALITY PROFILES
# =========================================================
# Persistent supplier behavior.
# This is VERY important for realism.
# Good suppliers stay mostly good.
# Bad suppliers continue attending management calls.
# =========================================================

supplier_quality_profiles = {}

for _, supplier in df_suppliers.iterrows():

    profile = np.random.choice(
        ["excellent", "good", "average", "poor"],
        p=[0.20, 0.45, 0.25, 0.10]
    )

    supplier_quality_profiles[
        supplier["supplier_id"]
    ] = profile


# =========================================================
# BASE DEFECT RATES
# =========================================================

quality_defect_rates = {

    "excellent": 0.003,
    "good": 0.015,
    "average": 0.050,
    "poor": 0.120
}


# =========================================================
# QUALITY INSPECTION GENERATION
# =========================================================

inspection_rows = []

inspection_counter = 1

for _, shipment in df_shipments.iterrows():

    # =====================================================
    # ONLY DELIVERED/DELAYED SHIPMENTS
    # =====================================================

    if shipment["shipment_status"] not in [
        "Delivered",
        "Delayed"
    ]:
        continue


    supplier_id = shipment["supplier_id"]


    # =====================================================
    # GET SUPPLIER RECORD
    # =====================================================

    supplier = df_suppliers[
        df_suppliers["supplier_id"] == supplier_id
    ].iloc[0]


    supplier_category = supplier["supplier_category"]

    criticality = supplier["criticality_level"]

    supplier_country = supplier["country"]


    # =====================================================
    # GET ACTUAL PO LINKED TO SHIPMENT
    # =====================================================

    po_id = shipment["po_id"]

    po = df_po[
        df_po["po_id"] == po_id
    ].iloc[0]


    received_quantity = po["received_quantity"]

    order_value = po["order_value"]


    # =====================================================
    # SUPPLIER QUALITY PROFILE
    # =====================================================

    quality_profile = supplier_quality_profiles[
        supplier_id
    ]

    base_defect_rate = quality_defect_rates[
        quality_profile
    ]


    # =====================================================
    # INSPECTION PROBABILITY MODEL
    # =====================================================
    # Enterprise-grade risk-based inspection logic.
    # Not every shipment gets inspected.
    # Procurement departments enjoy selective paranoia.
    # =====================================================

    inspection_probability = 0.10


    # Higher criticality suppliers inspected more
    if criticality >= 4:
        inspection_probability += 0.30

    elif criticality == 3:
        inspection_probability += 0.15


    # Delayed shipments more likely inspected
    if shipment["shipment_status"] == "Delayed":
        inspection_probability += 0.20


    # Poor suppliers heavily inspected
    if quality_profile == "poor":
        inspection_probability += 0.35

    elif quality_profile == "average":
        inspection_probability += 0.15

    elif quality_profile == "excellent":
        inspection_probability -= 0.05


    # Large financial exposure
    if order_value > 500000:
        inspection_probability += 0.10


    # International suppliers slightly higher risk
    if supplier_country != "India":
        inspection_probability += 0.05


    # Cap realistic probabilities
    inspection_probability = min(
        max(inspection_probability, 0.02),
        0.95
    )


    # =====================================================
    # SKIP SOME SHIPMENTS
    # =====================================================

    if np.random.rand() > inspection_probability:
        continue


    # =====================================================
    # INSPECTION DATE
    # =====================================================

    actual_arrival = pd.to_datetime(
        shipment["actual_arrival"]
    )

    inspection_date = (
        actual_arrival +
        timedelta(days=np.random.randint(0, 5))
    )


    # =====================================================
    # INSPECTED UNITS
    # =====================================================
    # Critical suppliers get deeper inspection.
    # =====================================================

    inspection_ratio = {

        1: np.random.uniform(0.05, 0.15),
        2: np.random.uniform(0.10, 0.25),
        3: np.random.uniform(0.20, 0.50),
        4: np.random.uniform(0.50, 0.80),
        5: np.random.uniform(0.80, 1.00)

    }[criticality]


    inspected_units = int(
        received_quantity * inspection_ratio
    )

    inspected_units = max(
        inspected_units,
        1
    )


    # =====================================================
    # DEFECT RATE ADJUSTMENTS
    # =====================================================

    adjusted_defect_rate = base_defect_rate


    # Delayed shipment correlation
    if shipment["shipment_status"] == "Delayed":
        adjusted_defect_rate *= 1.50


    # Sea shipments slightly higher damage risk
    if shipment["transport_mode"] == "sea":
        adjusted_defect_rate *= 1.20


    # Air freight generally safer
    elif shipment["transport_mode"] == "air":
        adjusted_defect_rate *= 0.90


    # High criticality suppliers often tighter quality
    if criticality >= 4:
        adjusted_defect_rate *= 0.85


    # Add random operational variance
    adjusted_defect_rate *= np.random.uniform(
        0.80,
        1.20
    )


    # =====================================================
    # REJECTED UNITS
    # =====================================================

    rejected_units = int(
        inspected_units * adjusted_defect_rate
    )


    # Add randomness
    rejected_units += np.random.randint(
        0,
        max(2, int(inspected_units * 0.02))
    )


    rejected_units = min(
        rejected_units,
        inspected_units
    )

    rejected_units = max(
        rejected_units,
        0
    )


    # =====================================================
    # DEFECT TYPE
    # =====================================================

    if rejected_units == 0:

        defect_type = "None"

    else:

        defect_type = random.choice(

            defect_map.get(
                supplier_category,
                ["General Defect"]
            )
        )


    # =====================================================
    # SEVERITY LOGIC
    # =====================================================

    rejection_pct = (
        rejected_units / inspected_units
    )


    if rejected_units == 0:

        severity = "Low"

    elif rejection_pct < 0.01:

        severity = "Low"

    elif rejection_pct < 0.05:

        severity = "Medium"

    elif rejection_pct < 0.10:

        severity = "High"

    else:

        severity = "Critical"


    # Electronics severe defects escalation
    if (
        supplier_category == "electronics"
        and defect_type in [
            "Voltage Instability",
            "ESD Damage"
        ]
        and rejected_units > 0
    ):

        severity = random.choice(
            ["High", "Critical"]
        )


    # =====================================================
    # CREATE RECORD
    # =====================================================

    inspection_rows.append({

        "inspection_id":
            f"INS{inspection_counter:07}",

        "shipment_id":
            shipment["shipment_id"],

        "po_id":
            po_id,

        "supplier_id":
            supplier_id,

        "inspection_date":
            inspection_date.date(),

        "inspected_units":
            inspected_units,

        "rejected_units":
            rejected_units,

        "defect_type":
            defect_type,

        "severity":
            severity
    })


    inspection_counter += 1


# =========================================================
# FINAL DATAFRAME
# =========================================================

df_quality_inspections = pd.DataFrame(
    inspection_rows
)


# =========================================================
# OPTIONAL VALIDATION CHECKS
# =========================================================

print("\n==============================")
print("QUALITY INSPECTION SUMMARY")
print("==============================")

print(
    f"Total inspections: "
    f"{len(df_quality_inspections)}"
)

print(
    "\nSeverity Distribution:"
)

print(
    df_quality_inspections[
        "severity"
    ].value_counts()
)

print(
    "\nDefect Distribution:"
)

print(
    df_quality_inspections[
        "defect_type"
    ].value_counts().head(10)
)

print(
    "\nInspection Coverage %:"
)

coverage = (
    len(df_quality_inspections)
    / len(df_shipments)
) * 100

print(
    round(coverage, 2),
    "%"
)

print("\nSample Records:\n")

print(
    df_quality_inspections.head()
)

In [ ]:

df_quality_inspections.head()


In [ ]:
df_quality_inspections = df_quality_inspections.merge(
    df_shipments[["shipment_id", "actual_arrival"]],
    on="shipment_id",
    how="left"
)

df_quality_inspections.loc[
    df_quality_inspections["actual_arrival"].isna(),
    "inspection_date"
] = pd.NaT

df_quality_inspections.drop(columns=["actual_arrival"], inplace=True)

In [ ]:
df_quality_inspections.info()

In [ ]:
df_quality_inspections.to_csv("data/raw/df_quality_inspections.csv", index=False)

In [ ]:
problem_records = (
    df_quality_inspections
    .merge(
        df_shipments[["shipment_id", "shipment_status"]],
        on="shipment_id",
        how="left"
    )
    .loc[lambda df: ~df["shipment_status"].isin(["Delivered", "Delayed"])]
)

problem_records

In [ ]:
unique_shipment_status = (
    df_quality_inspections
    .merge(
        df_shipments[["shipment_id", "shipment_status"]],
        on="shipment_id",
        how="left"
    )["shipment_status"]
    .dropna()
    .unique()
)

unique_shipment_status